# Tokenization — Hands-On

**LLM Engineering · Domain 1 · Roadmap Week 09**

Companion to `02 Literature Notes/LLM Engineering/Tokenization` and the deck
`Lesson_02_Tokenization.pptx`. Runs fully offline (no API key).

**Sources:** OpenAI tiktoken docs; Sennrich et al. BPE (arXiv:1508.07909);
Karpathy 'Let's build the GPT Tokenizer'; HuggingFace Tokenizers.

## 0. Setup

In [ ]:
%pip install -q tiktoken
import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # gpt-4o / gpt-4.1 family
print("vocab pieces:", enc.n_vocab)

## 1. Encode / decode is lossless and deterministic
A tokenizer maps text to integer IDs and back with an exact round-trip.

In [ ]:
text = "Tokenization drives cost."
ids = enc.encode(text)
print("ids   :", ids)
print("pieces:", [enc.decode([i]) for i in ids])
print("decode:", enc.decode(ids))
assert enc.decode(ids) == text
print("round-trip OK")

## 2. See subword boundaries and compression
Watch how one 'word' becomes several subword pieces, and how the chars/token
compression ratio changes with language and content.

In [ ]:
def show(t):
    ids = enc.encode(t)
    ratio = len(t)/max(len(ids),1)
    print(f"{len(ids):>3} tok  {ratio:>4.2f} c/t  {t!r} -> {[enc.decode([i]) for i in ids]}")

for t in ["internationalization", " token", "token", "1234567890",
          "https://example.com/path?q=1", "東京へようこそ",
          "def add(a, b):\n    return a + b"]:
    show(t)

**Observe:** ` token` (leading space) and `token` get *different* IDs;
numbers, URLs, non-English, and code all fragment — costing more tokens per idea.

## 3. The chars≈4/token rule of thumb — and where it breaks

In [ ]:
samples = {
  "English prose": "The quick brown fox jumps over the lazy dog near the river bank.",
  "Code":          "for i in range(10):\n    print(i*i)",
  "Japanese":      "私は毎朝コーヒーを飲みます。",
  "Digits":        "3.14159265358979323846",
}
for name, t in samples.items():
    ids = enc.encode(t)
    print(f"{name:14s}: {len(t):3d} chars / {len(ids):3d} tokens = {len(t)/len(ids):.2f} c/t")

## 4. Count tokens for a chat request (with template overhead)
A naive sum of message lengths undercounts — each message carries wrapping tokens.

In [ ]:
def count_message_tokens(messages, model="gpt-4o-mini"):
    try: e = tiktoken.encoding_for_model(model)
    except KeyError: e = tiktoken.get_encoding("o200k_base")
    return sum(3 + len(e.encode(m["content"])) for m in messages) + 3

messages = [
  {"role":"system","content":"You are a concise assistant."},
  {"role":"user","content":"Summarize the attached contract clause in one line."},
]
naive = sum(len(enc.encode(m["content"])) for m in messages)
print("naive content tokens :", naive)
print("with chat overhead   :", count_message_tokens(messages))

## 5. Dollar cost estimation (the FDE utility)
Input tokens are exact; output tokens are estimated. Scale by request volume.

In [ ]:
PRICING = {"gpt-4o-mini": {"in":0.15,"out":0.60}, "gpt-4o": {"in":2.50,"out":10.00}}

def estimate(messages, model="gpt-4o-mini", out_tokens=400, calls=1):
    n_in = count_message_tokens(messages, model); p = PRICING[model]
    per_call = (n_in*p["in"] + out_tokens*p["out"]) / 1_000_000
    return {"in":n_in, "out":out_tokens,
            "per_call_$":round(per_call,6), "at_%d_calls_$"%calls:round(per_call*calls,2)}

print(estimate(messages, "gpt-4o-mini", out_tokens=400, calls=100_000))
print(estimate(messages, "gpt-4o",      out_tokens=400, calls=100_000))

## 6. Why LLMs struggle with character-exact tasks
Characters are hidden inside merged tokens, so the model cannot 'see' them.

In [ ]:
word = "strawberry"
ids = enc.encode(word)
print("tokens:", [enc.decode([i]) for i in ids])
print("The letter 'r' spans merged tokens -> counting/reversing letters is hard for the model.")

## 7. Exercises
1. Find a word that tokenizes to 1 token and one that tokenizes to 5+. What differs?
2. Compare `cl100k_base` vs `o200k_base` token counts on the code sample. Which is cheaper?
3. Add a 3rd message to §4 and predict the count before running.
4. Estimate monthly cost for 1,200-in / 600-out at 500k calls on both models.
5. Encode an emoji and a rare surname — how many byte-level pieces result?

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Tokenization`
- Snippets: `04 Code Snippets/LLM/Inspecting a Tokenizer with tiktoken`, `... / Token Counting and Cost Estimation`
- MOC: `06 Maps of Content/LLM Engineering Concepts`